# OpenBCI Ganglion 실습 2 · 실시간 손동작 분류

**재활재생개론 · 근전도(EMG)로 가위·바위·보 알아맞히기**

---

## 이 실습에서 하는 일

1차시에서는 신호를 **보는** 법을 배웠습니다.
2차시에서는 신호로 **판단하는** 법을 배웁니다.

```
팔 근육 EMG  →  특징 추출  →  머신러닝(Random Forest)  →  가위/바위/보 판정
```

| 단계 | 내용 |
| --- | --- |
| 1–3 | 준비 · 연결 · 전극 부착 |
| 4 | 실시간 신호 확인 (힘줘 보기) |
| 5–6 | 필터 설계, 특징 추출 |
| 7 | 캘리브레이션 — 세 동작을 40번씩 |
| 8 | 근육 활성 지문 (동작이 구분되나 확인) |
| 9–10 | 모델 학습, 성능 평가 |
| 11 | 모델 저장 |
| 12 | 실시간 판정 |

## 준비물

- **OpenBCI Ganglion 보드** + **BLED112 동글**
- **EMG 전극** (팔뚝에 붙일 수 있는 것)
- 1차시를 먼저 완료했을 것 (연결이 된다는 전제)

## EEG 가 아니라 EMG 입니다

1차시는 머리에서 **뇌파(EEG)** 를 봤지만, 2차시는 팔뚝에서 **근전도(EMG)** 를 봅니다.
같은 Ganglion 보드로 둘 다 측정하지만, **관심 주파수 대역이 다릅니다.**

| | 대역 | 이유 |
| --- | --- | --- |
| EEG (1차시) | 1–45 Hz | 뇌파는 느린 리듬 |
| **EMG (2차시)** | **30–95 Hz** | 근육 활동은 더 빠른 성분 |

---

> ⚠️ **1차시와 마찬가지로 OpenBCI GUI 는 반드시 종료하세요.**

**위에서 아래로 셀을 하나씩 실행하세요.** (`Shift + Enter`)

---
# 1단계 · 라이브러리 준비

1차시와 같은 준비에 더해, 이번에는 머신러닝 도구(**scikit-learn**)를 씁니다.

> brainflow 는 반드시 5.20.0 이어야 합니다 (1차시에서 다룬 내용).
> 버전을 바꿨다면 **커널을 재시작**하세요.

In [ ]:
import sys
import subprocess
import warnings
from importlib.metadata import version as _pkg_version, PackageNotFoundError

# 콘솔에서 한글과 µ 기호가 깨지지 않도록 (Windows cp949 대응)
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

warnings.filterwarnings('ignore', message='pkg_resources is deprecated')

# ─────────────────────────────────────────────────────────────
# brainflow 는 반드시 5.20.0 이어야 한다 (BLED112 동글 지원).
# 디스크 버전과 메모리에 로드된 DLL 버전을 모두 확인한다.
# ─────────────────────────────────────────────────────────────
REQUIRED_BF = '5.20.0'
_need_restart = False

try:
    _disk_bf = _pkg_version('brainflow')
except PackageNotFoundError:
    _disk_bf = None

# 필요한 패키지 설치
_to_install = []
if _disk_bf != REQUIRED_BF:
    _to_install.append(f'brainflow=={REQUIRED_BF}')
for _pkg, _imp in [('scikit-learn', 'sklearn'), ('joblib', 'joblib')]:
    try:
        __import__(_imp)
    except ImportError:
        _to_install.append(_pkg)

if _to_install:
    print('설치 중:', ', '.join(_to_install))
    print('1~2분 걸릴 수 있습니다...')
    try:
        import pkg_resources
    except ImportError:
        _to_install.append('setuptools<81')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_to_install])
    print('설치 완료')
    if 'brainflow' in sys.modules and _disk_bf != REQUIRED_BF:
        _need_restart = True

# 메모리에 로드된 brainflow DLL 버전 확인
if 'brainflow' in sys.modules and not _need_restart:
    try:
        from brainflow.board_shim import BoardShim as _BS
        if _BS.get_version() != REQUIRED_BF:
            _need_restart = True
    except Exception:
        _need_restart = True

if _need_restart:
    print()
    print('!' * 62)
    print('  커널을 재시작해야 합니다.  (Kernel > Restart Kernel)')
    print('  그 다음 이 셀부터 다시 실행하세요.')
    print('!' * 62)
    raise SystemExit

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from collections import deque, Counter

from scipy import signal as sp_signal
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import joblib

from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds

# 그래프 한글 표시 - 설치된 한글 폰트 중 먼저 발견되는 것을 사용
from matplotlib import font_manager
_installed_fonts = {f.name for f in font_manager.fontManager.ttflist}
_korean_font = '(없음)'
for _cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans KR', 'Gulim']:
    if _cand in _installed_fonts:
        plt.rcParams['font.family'] = _cand
        _korean_font = _cand
        break
plt.rcParams['axes.unicode_minus'] = False

import sklearn
print('준비 완료')
print(f'  Python       : {sys.version.split()[0]}')
print(f'  BrainFlow    : {BoardShim.get_version()}   (실제 로드된 버전)')
print(f'  scikit-learn : {sklearn.__version__}')
print(f'  한글 폰트    : {_korean_font}')

---
# 2단계 · 동글 포트 확인

1차시와 동일합니다. 동글이 꽂힌 COM 포트를 확인합니다.

In [ ]:
from serial.tools import list_ports

ports = list(list_ports.comports())
print('연결된 COM 포트')
print('=' * 60)
if not ports:
    print('포트를 찾지 못했습니다. BLED112 동글을 꽂았는지 확인하세요.')
else:
    for p in ports:
        desc = (p.description or '').lower()
        hint = '  <-- 동글일 가능성 높음' if ('bluetooth' in desc or 'cp210' in desc) else ''
        print(f'  {p.device:<8} {p.description}{hint}')
print('=' * 60)

---
# 3단계 · 설정 ⭐ **여기를 고치세요**

1차시와 같은 설정에 더해, 이번에는 **분류할 동작**을 정합니다.

In [ ]:
# ===================== 학생 설정 구역 =====================

COM_PORT  = 'COM3'        # 동글 포트 (2단계에서 확인)
BOARD_MAC = ''            # 비워 두세요.
                          # BLED112 동글 방식은 MAC 을 지정해도 무시됩니다.
                          # 여러 보드 대처법은 아래 스캔 셀 설명 참고

# 분류할 손동작 (라벨 번호 : 이름)
GESTURES = {
    1: 'Rock',       # 주먹
    2: 'Scissor',    # 가위
    3: 'Paper',      # 보
}

# 캘리브레이션: 각 동작을 몇 번 반복할지
N_REPEAT = 40            # 동작당 샘플 수 (원본과 동일)

# ==========================================================

BOARD_ID = BoardIds.GANGLION_BOARD
FS       = BoardShim.get_sampling_rate(BOARD_ID)         # 200 Hz
EEG_CH   = BoardShim.get_eeg_channels(BOARD_ID)          # EMG 로 사용할 4채널
N_CH     = len(EEG_CH)

WINDOW_SAMPLES = 100     # 한 번에 분석할 창 크기 (100샘플 = 0.5초)

print('설정 확인')
print('=' * 60)
print(f'  동글 포트     : {COM_PORT}')
print(f'  대상 보드     : {BOARD_MAC if BOARD_MAC else "(자동 탐색)"}')
print(f'  샘플링        : {FS} Hz,  채널 {N_CH}개 {EEG_CH}')
print(f'  분석 창       : {WINDOW_SAMPLES} 샘플 ({WINDOW_SAMPLES/FS:.2f}초)')
print(f'  동작          : {list(GESTURES.values())}')
print(f'  동작당 반복   : {N_REPEAT} 회')
print('=' * 60)

---
# 🔎 주변 Ganglion 확인 · 다중 보드 대처

**강의실처럼 Ganglion 이 여러 대 켜져 있으면 반드시 읽으세요.**

### ⚠️ 먼저 알아야 할 사실

BLED112 동글 방식에서는 **`BOARD_MAC` 을 지정해도 보드가 선택되지 않습니다.**
BrainFlow 가 값을 받기는 하지만 실제 필터링에 쓰지 않아, **먼저 응답한 보드에 연결**됩니다.
(존재하지 않는 가짜 MAC 을 넣어도 그냥 연결되는 것으로 확인됨 · 2026-07-24 실측)

> 그래서 여러 대가 켜져 있으면 **옆 사람 보드에 붙을 수 있습니다.**

### ✅ 실제로 통하는 방법 — 순서대로 연결

**한 번에 한 대씩만 켜고 연결합니다.**

1. 모든 보드 전원 **OFF**
2. 첫 번째 학생: 보드 ON → 연결(다음 단계) → 연결되면 그대로 유지
3. 두 번째 학생: 보드 ON → 연결 → …
4. 이후 반복

BLE 보드는 **연결되고 나면 광고를 멈춥니다.**
따라서 이미 연결된 보드는 다음 사람에게 잡히지 않습니다.

### ✅ 두 번째 안전장치 — 내 보드인지 확인

연결한 뒤 **팔에 힘을 줘 보세요.** 신호가 반응하면 내 보드입니다.
아무 반응이 없으면 남의 보드일 수 있으니, **연결을 해제하고 다시 시도**하세요.

---

아래 셀은 **주변에 Ganglion 이 몇 대 켜져 있는지** 확인합니다.
1대뿐이면 안심하고 연결해도 되고, 여러 대면 위 절차를 따라야 합니다.

> ⚠️ **보드에 연결되지 않은 상태에서 실행하세요.** 동글을 직접 쓰기 때문에
> 이미 연결 중이면 실패합니다.

In [ ]:
import serial

def scan_ble_devices(port, duration=8.0):
    """BLED112 동글로 주변 BLE 장치를 스캔해 MAC 과 이름을 얻는다.

    BrainFlow 는 발견한 장치의 MAC 을 알려주지 않으므로,
    동글에 BGAPI 명령을 직접 보내 광고(advertisement) 패킷을 읽는다.

    반환: {mac: {'name': 이름 또는 None, 'rssi': 신호세기}}
    """
    def pkt(method, payload=b''):
        return bytes([0x00, len(payload), 0x06, method]) + payload   # class 0x06 = GAP

    def parse_name(data):
        i = 0
        while i < len(data):
            ln = data[i]
            if ln == 0:
                break
            if i + 1 < len(data) and data[i + 1] in (0x08, 0x09):     # 단축/완전 이름
                name = data[i + 2:i + 1 + ln].decode('utf-8', errors='ignore')
                name = name.strip('\x00').strip()
                return name if name else None
            i += ln + 1
        return None

    found = {}
    with serial.Serial(port, 115200, timeout=0.2) as ser:
        ser.write(pkt(0x04))                                          # 진행중 절차 종료
        time.sleep(0.2); ser.reset_input_buffer()
        ser.write(pkt(0x07, bytes([0x4B, 0x00, 0x32, 0x00, 0x01])))   # 스캔 파라미터
        time.sleep(0.2); ser.reset_input_buffer()
        ser.write(pkt(0x02, bytes([0x02])))                           # 스캔 시작

        buf = bytearray()
        t_end = time.time() + duration
        while time.time() < t_end:
            chunk = ser.read(256)
            if chunk:
                buf.extend(chunk)
            while len(buf) >= 4:
                plen = ((buf[0] & 0x07) << 8) | buf[1]
                total = 4 + plen
                if len(buf) < total:
                    break
                p = bytes(buf[:total]); del buf[:total]
                if (p[0] & 0x80) and p[2] == 0x06 and p[3] == 0x00 and plen >= 11:
                    body = p[4:]
                    rssi = body[0] - 256 if body[0] > 127 else body[0]
                    mac  = ':'.join(f'{b:02x}' for b in reversed(body[2:8]))
                    name = parse_name(body[11:11 + body[10]])
                    e = found.setdefault(mac, {'name': None, 'rssi': -999})
                    if name:
                        e['name'] = name
                    e['rssi'] = max(e['rssi'], rssi)

        ser.write(pkt(0x04))                                          # 스캔 종료
        time.sleep(0.1)
    return found


print(f'{COM_PORT} 에서 8초간 스캔합니다. 잠시 기다리세요...')
devices = scan_ble_devices(COM_PORT, 8.0)

ganglions = {m: i for m, i in devices.items()
             if i['name'] and 'ganglion' in i['name'].lower()}

print()
print('=' * 64)
print(f'주변 Ganglion {len(ganglions)}대 감지')
print('=' * 64)

if ganglions:
    for mac, info in sorted(ganglions.items(), key=lambda x: -x[1]['rssi']):
        near = '  <- 가장 가까움' if mac == max(ganglions, key=lambda m: ganglions[m]['rssi']) else ''
        print(f'  {info["name"]:<20} {mac}   신호 {info["rssi"]:4d} dBm{near}')
    print()
    if len(ganglions) == 1:
        print('1대뿐입니다. 바로 연결해도 안전합니다.')
    else:
        print(f'{len(ganglions)}대가 켜져 있습니다. 주의가 필요합니다.')
        print()
        print('  * BOARD_MAC 을 지정해도 보드는 선택되지 않습니다.')
        print('    (BrainFlow 의 BLED112 경로가 MAC 을 무시함)')
        print()
        print('  대처 방법')
        print('    1) 내 보드만 남기고 나머지는 전원을 끈다  <- 가장 확실')
        print('    2) 또는 한 대씩 순서대로 연결한다')
        print('       (연결된 보드는 광고를 멈춰 다음 사람에게 안 잡힘)')
        print('    3) 연결 후 팔에 힘을 줘서 신호가 반응하는지 꼭 확인한다')
else:
    print('  Ganglion 을 찾지 못했습니다.')
    print()
    print('  확인할 것')
    print('    1. 보드 전원 스위치 ON (파란 LED 깜빡임)')
    print('    2. 배터리 충전 상태')
    print('    3. 이미 보드에 연결 중이면 먼저 연결을 해제할 것')
    print('    4. OpenBCI GUI 종료')

print()
print(f'(주변 BLE 장치 총 {len(devices)}개 감지 - Ganglion 외 기기 포함)')

---
# 4단계 · 보드 연결

1차시와 같은 연결 코드입니다. 최대 3회 자동 재시도합니다.

> ⚠️ OpenBCI GUI 종료 확인. 전극을 팔에 붙이고 보드에 연결한 상태여야 합니다.

In [ ]:
BoardShim.disable_board_logger()

params = BrainFlowInputParams()
params.serial_port = COM_PORT
params.mac_address = BOARD_MAC
params.timeout     = 40

board     = BoardShim(BOARD_ID, params)
connected = False
MAX_TRY   = 3

print(f'{COM_PORT} 를 통해 Ganglion 연결 시도 중...')
print('첫 시도는 40초까지 걸릴 수 있습니다.')
print()

for attempt in range(1, MAX_TRY + 1):
    t0 = time.time()
    try:
        board.prepare_session()
        connected = True
        print(f'[{attempt}/{MAX_TRY}] 연결 성공 ({time.time()-t0:.1f}초)')
        break
    except Exception as e:
        print(f'[{attempt}/{MAX_TRY}] 실패 ({time.time()-t0:.1f}초): {e}')
        try:
            board.release_session()
        except Exception:
            pass
        if attempt < MAX_TRY:
            print('        다시 시도합니다...')
            time.sleep(3)

print()
if connected:
    board.start_stream(450000, '')
    print('스트리밍 시작. 5단계로 진행하세요.')
else:
    print('연결 실패. 1차시의 문제 해결 표와 5-A 진단 셀을 참고하세요.')
    print('가장 흔한 원인: OpenBCI GUI 미종료, brainflow 버전, 커널 재시작.')

---
# 5단계 · 실시간 신호 확인  🚦 **관문 1**

**본격적인 수집 전에, 전극이 제대로 붙었는지 먼저 확인합니다.**
여기서 문제를 잡지 않으면, 뒤에서 40×3회를 수집한 뒤에야 실패를 알게 됩니다.

아래 셀을 실행하면 10초간 실시간 파형이 갱신됩니다.

> 🧪 **이렇게 확인하세요**: 실행하는 동안 **주먹을 꽉 쥐었다 폈다** 반복하세요.
> 힘줄 때 파형(특히 특정 채널)이 **눈에 띄게 커져야** 합니다.
> 아무 반응이 없으면 → 전극 접촉 불량. 전극을 다시 붙이고 이 셀을 다시 실행하세요.

### EMG 전극 부착 (팔뚝)

- 4개 채널 전극을 **팔뚝의 서로 다른 근육 위**에 붙입니다
- 레퍼런스(기준) 전극은 **뼈 위**(손목 안쪽 등 근육이 적은 곳)에
- 동작마다 다른 근육이 쓰이도록 골고루 배치하는 것이 핵심

In [ ]:
if not connected:
    print('보드가 연결되지 않았습니다. 4단계를 먼저 성공시키세요.')
else:
    from IPython.display import clear_output

    print('10초간 실시간 신호. 주먹을 쥐었다 폈다 해 보세요.')
    n_frames = 20
    for frame in range(n_frames):
        time.sleep(0.5)
        d = board.get_current_board_data(WINDOW_SAMPLES)
        if d.shape[1] < 2:
            continue
        emg = d[EEG_CH, :]

        clear_output(wait=True)
        fig, axes = plt.subplots(N_CH, 1, figsize=(11, 6), sharex=True)
        fig.suptitle(f'실시간 EMG  ({frame+1}/{n_frames})  — 힘줄 때 커지는지 보세요',
                     fontweight='bold')
        for i in range(N_CH):
            axes[i].plot(emg[i], linewidth=0.8)
            axes[i].set_ylabel(f'Ch{i+1}')
            axes[i].grid(alpha=0.3)
        axes[-1].set_xlabel('샘플')
        plt.tight_layout()
        plt.show()

    print('확인 완료. 힘줄 때 반응이 있었다면 6단계로.')
    print('반응이 없었다면 전극을 다시 붙이고 이 셀을 다시 실행하세요.')

---
# 6단계 · 필터와 특징 추출

원시 EMG를 그대로 머신러닝에 넣지 않습니다. 두 단계를 거칩니다.

### ① 필터
- **60Hz 노치**: 전원 잡음 제거
- **30–95Hz 대역통과**: EMG 성분만 남김

> 원본 MATLAB은 30–**100**Hz였지만, 200Hz 샘플링에서 표현 가능한 최대 주파수(나이퀴스트)가
> 정확히 100Hz입니다. 경계값은 필터가 불안정해지므로 **95Hz**로 둡니다.
> 100Hz 성분은 어차피 거의 없어 손실이 없습니다.

### ② 특징 추출 — 4채널을 숫자 4개로
각 채널의 **포락선(envelope) 평균**을 구합니다.
포락선은 신호의 "세기 윤곽"으로, `|Hilbert 변환|`으로 얻습니다.

즉 **"각 근육이 지금 얼마나 활성인가"** 를 채널당 숫자 하나로 요약합니다.
동작마다 쓰는 근육이 다르므로, 이 4개 숫자의 패턴으로 동작을 구별할 수 있습니다.

In [ ]:
# 필터 계수 (한 번만 설계)
NOTCH_FREQ = 60.0
BAND       = (30.0, 95.0)   # 30-95 Hz (나이퀴스트 100Hz 회피)

_nyq = FS / 2
b_notch, a_notch = sp_signal.iirnotch(NOTCH_FREQ / _nyq, Q=30)
b_band,  a_band  = sp_signal.butter(4, [BAND[0]/_nyq, BAND[1]/_nyq], btype='band')


def extract_features(window):
    """원시 EMG 창 -> 특징 벡터.

    window: (samples, n_channels) 형태의 원시 EMG
    반환   : (n_channels,) — 채널별 포락선 평균
    """
    x = sp_signal.filtfilt(b_notch, a_notch, window, axis=0)   # 노치
    x = sp_signal.filtfilt(b_band,  a_band,  x,      axis=0)   # 대역통과
    envelope = np.abs(sp_signal.hilbert(x, axis=0))            # 포락선
    return envelope.mean(axis=0)                               # 채널당 평균


print('필터와 특징 추출 함수 준비 완료')
print(f'  노치      : {NOTCH_FREQ} Hz')
print(f'  대역통과  : {BAND[0]}-{BAND[1]} Hz')
print(f'  특징      : 채널당 포락선 평균 -> {N_CH}개')

---
# 7단계 · 캘리브레이션 (데이터 수집)

이제 각 동작을 하고 있는 동안의 EMG를 모읍니다.
동작 하나당 **40번**, 세 동작이면 120개의 특징 벡터가 모입니다.

**진행 방식** (동작마다):
1. 화면에 "Make Rock" 안내 + 3초 카운트다운
2. 그 자세를 **계속 유지**한 채로 40번 수집 (약 10초)
3. 다음 동작으로

> 🧪 수집 중에는 **자세를 유지하고 힘의 세기도 일정하게** 유지하세요.
> 흔들리면 나중에 모델이 헷갈립니다.

> ⚠️ 이 셀은 약 40초 걸립니다. 중간에 멈추지 마세요.

In [ ]:
if not connected:
    print('보드가 연결되지 않았습니다.')
else:
    def collect_gesture(name, n_repeat):
        """한 동작을 n_repeat 번 수집해서 특징 배열 반환"""
        print(f'\n>>> 준비: {name}')
        for c in (3, 2, 1):
            print(f'    {c}...')
            time.sleep(1)
        print(f'    시작! {name} 자세를 유지하세요')

        feats = np.zeros((n_repeat, N_CH))
        for i in range(n_repeat):
            d = board.get_current_board_data(WINDOW_SAMPLES)
            feats[i] = extract_features(d[EEG_CH, :].T)   # (samples, ch)
            time.sleep(0.25)
            if (i + 1) % 10 == 0:
                print(f'    {name}: {(i+1)/n_repeat*100:.0f}%')
        return feats

    # 버퍼를 비우고 시작
    board.get_board_data()

    X_list, Y_list = [], []
    for label, name in GESTURES.items():
        feats = collect_gesture(name, N_REPEAT)
        X_list.append(feats)
        Y_list.append(np.full(N_REPEAT, label))

    X = np.vstack(X_list)          # (동작수 * N_REPEAT, N_CH)
    Y = np.concatenate(Y_list)

    print('\n캘리브레이션 완료')
    print(f'  특징 행렬 : {X.shape}')
    print(f'  라벨      : {Y.shape}')

---
# 8단계 · 근육 활성 지문  🚦 **관문 2**

수집한 데이터로 각 동작의 **평균 활성 패턴**을 그립니다.
네 방향은 네 채널, 바깥으로 뻗을수록 그 근육이 활성입니다.

**세 그림의 모양이 서로 뚜렷하게 다르면** 성공입니다.
모양이 비슷하게 겹쳐 보이면, 머신러닝도 구별하기 어렵습니다.
→ 전극 위치를 바꿔 다른 근육을 잡도록 하고 7단계부터 다시 하세요.

In [ ]:
if 'X' not in dir():
    print('먼저 7단계로 데이터를 수집하세요.')
else:
    fig, axes = plt.subplots(1, len(GESTURES), figsize=(4*len(GESTURES), 4),
                             subplot_kw={'projection': 'polar'})
    if len(GESTURES) == 1:
        axes = [axes]

    angles = np.linspace(0, 2*np.pi, N_CH, endpoint=False)
    angles = np.concatenate([angles, angles[:1]])   # 닫힌 도형

    global_max = 0
    means = {}
    for label, name in GESTURES.items():
        m = X[Y == label].mean(axis=0)
        means[label] = m
        global_max = max(global_max, m.max())

    for ax, (label, name) in zip(axes, GESTURES.items()):
        vals = np.concatenate([means[label], means[label][:1]])
        ax.plot(angles, vals, linewidth=2)
        ax.fill(angles, vals, alpha=0.25)
        ax.set_title(name, fontweight='bold', pad=15)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([f'Ch{i+1}' for i in range(N_CH)])
        ax.set_ylim(0, global_max * 1.1)

    plt.tight_layout()
    plt.show()

    print('세 지문의 모양이 서로 다른가요?')
    print('  다르다  -> 9단계로 진행')
    print('  비슷하다 -> 전극 위치를 바꿔 7단계부터 다시')

---
# 9단계 · Random Forest 학습 & 정직한 평가

**Random Forest**는 여러 개의 결정 트리가 다수결로 판정하는 모델입니다.
특징 4개로 동작을 맞히도록 학습시킵니다.

### 시간 순서로 나눕니다 (중요)
데이터를 **섞지 않고** 각 동작의 앞 80%로 학습, 뒤 20%로 시험합니다.

> 왜 안 섞나? 연속으로 수집한 이웃 샘플은 거의 똑같습니다.
> 무작위로 섞으면 거의 같은 샘플이 학습셋과 시험셋에 나뉘어 들어가,
> 시험이 **너무 쉬워지고 정확도가 부풀려집니다.** 시간순으로 나누면
> "새로운 순간"을 맞히는 셈이라 실제 성능에 가깝습니다.
> (정확도가 100%가 아니어도 정상입니다 — 오히려 정직한 숫자입니다.)

In [ ]:
if 'X' not in dir():
    print('먼저 7단계로 데이터를 수집하세요.')
else:
    # 각 동작별로 앞 80% 학습 / 뒤 20% 시험 (시간순, 섞지 않음)
    train_idx, test_idx = [], []
    for label in GESTURES:
        idx = np.where(Y == label)[0]        # 이 동작의 인덱스 (시간순)
        cut = int(len(idx) * 0.8)
        train_idx.extend(idx[:cut])
        test_idx.extend(idx[cut:])
    train_idx = np.array(train_idx)
    test_idx  = np.array(test_idx)

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X[train_idx], Y[train_idx])

    Y_pred = clf.predict(X[test_idx])
    accuracy = (Y_pred == Y[test_idx]).mean() * 100

    print(f'학습 샘플 : {len(train_idx)}개')
    print(f'시험 샘플 : {len(test_idx)}개')
    print(f'시험 정확도 : {accuracy:.1f}%')
    print()

    # 혼동행렬 + 특징 중요도
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    labels_sorted = sorted(GESTURES)
    names_sorted = [GESTURES[l] for l in labels_sorted]
    cm = confusion_matrix(Y[test_idx], Y_pred, labels=labels_sorted)
    disp = ConfusionMatrixDisplay(cm, display_labels=names_sorted)
    disp.plot(ax=ax1, colorbar=False)
    ax1.set_title(f'혼동행렬 (정확도 {accuracy:.1f}%)', fontweight='bold')

    ax2.bar([f'Ch{i+1}' for i in range(N_CH)], clf.feature_importances_)
    ax2.set_title('채널별 중요도', fontweight='bold')
    ax2.set_ylabel('중요도')
    ax2.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

    print('혼동행렬의 대각선이 진할수록 잘 맞힌 것입니다.')
    print('특정 채널 중요도가 0에 가까우면 그 전극은 도움이 안 되고 있습니다.')

---
# 10단계 · 모델 저장

학습된 모델을 파일로 저장합니다. **모델과 함께 설정 정보(메타데이터)** 를 같이 넣습니다.
이게 없으면 나중에 파일만 보고 무엇으로 학습했는지 알 수 없습니다.

### 왜 저장하나 — 캘리브레이션은 매번 하는데?
- **다음 주에 지난주 모델 불러와 비교**: 전극 위치가 바뀌면 성능이 어떻게 달라지나
- **친구 모델로 내 신호 판정**: 왜 남의 모델은 나한테 잘 안 맞을까? (개인차)
- 잘 나온 모델을 보관 → 재현

이것이 실제 BCI 연구의 핵심 주제입니다: **한 번 학습한 모델이 언제까지, 누구에게 통하는가.**

In [ ]:
if 'clf' not in dir():
    print('먼저 9단계로 모델을 학습하세요.')
else:
    out_dir = Path('..') / 'outputs'
    out_dir.mkdir(exist_ok=True)
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # 모델 + 메타데이터를 하나로 묶어 저장
    bundle = {
        'model':        clf,
        'gestures':     GESTURES,
        'fs':           FS,
        'channels':     N_CH,
        'window':       WINDOW_SAMPLES,
        'filter':       {'notch': NOTCH_FREQ, 'band': BAND},
        'accuracy':     float(accuracy),
        'n_repeat':     N_REPEAT,
        'date':         datetime.now().isoformat(timespec='seconds'),
    }

    model_path = out_dir / f'emg_model_{stamp}.joblib'
    joblib.dump(bundle, model_path)

    print('모델 저장 완료')
    print(f'  파일     : {model_path.resolve()}')
    print(f'  동작     : {list(GESTURES.values())}')
    print(f'  정확도   : {accuracy:.1f}%')
    print()
    print('나중에 불러오기:')
    print(f"  bundle = joblib.load(r'{model_path.name}')")
    print("  clf = bundle['model']")

---
# 11단계 · 실시간 판정

이제 학습한 모델로 **실시간으로** 손동작을 알아맞힙니다.
매 순간 신호를 받아 특징을 뽑고, 모델에 넣어 예측합니다.

### 다수결 안정화
매 순간의 예측은 값이 튑니다(잠깐 엉뚱한 답이 나옴).
그래서 **최근 5회 예측의 다수결**로 최종 판정을 냅니다. 훨씬 안정적입니다.

> 🧪 실행하는 동안 가위·바위·보를 바꿔 가며 취해 보세요.
> 화면의 판정이 따라오는지 확인하세요.

In [ ]:
if 'clf' not in dir():
    print('먼저 9단계로 모델을 학습하세요 (또는 10단계 저장 모델을 불러오세요).')
elif not connected:
    print('보드가 연결되지 않았습니다.')
else:
    from IPython.display import clear_output

    recent = deque(maxlen=5)     # 최근 5회 예측
    board.get_board_data()       # 버퍼 비우기

    print('실시간 판정 시작. 동작을 바꿔 보세요. (약 30초)')
    n_frames = 60
    for frame in range(n_frames):
        d = board.get_current_board_data(WINDOW_SAMPLES)
        if d.shape[1] < WINDOW_SAMPLES:
            time.sleep(0.25)
            continue

        feat = extract_features(d[EEG_CH, :].T).reshape(1, -1)
        pred = int(clf.predict(feat)[0])
        recent.append(pred)

        # 다수결
        stable = Counter(recent).most_common(1)[0][0]

        clear_output(wait=True)
        bar = {l: ('█' * int(v * 20)) for l, v in
               zip(clf.classes_, clf.predict_proba(feat)[0])}
        print(f'프레임 {frame+1}/{n_frames}')
        print('=' * 40)
        print(f'  순간 예측 : {GESTURES.get(pred, pred)}')
        print(f'  안정 판정 : >>> {GESTURES.get(stable, stable)} <<<')
        print('=' * 40)
        for l in sorted(GESTURES):
            p = clf.predict_proba(feat)[0][list(clf.classes_).index(l)]
            print(f'  {GESTURES[l]:8s} {"█"*int(p*20):<20} {p*100:4.0f}%')

        time.sleep(0.25)

    print('\n실시간 판정 종료.')

---
# 12단계 · 연결 해제  ⚠️ **꼭 실행하세요**

1차시와 동일합니다. 실행하지 않으면 동글이 계속 점유됩니다.

In [ ]:
try:
    if board.is_prepared():
        board.release_session()
        print('연결 해제 완료. 동글이 사용 가능한 상태입니다.')
    else:
        print('이미 해제되어 있습니다.')
except Exception as e:
    print(f'해제 중 문제 발생: {e}')
    print('노트북 커널을 재시작하면 정리됩니다.')

---
# 부록 · 저장한 모델 불러오기

다음 주에 이어서 하거나, 친구 모델을 시험할 때 사용합니다.
이 셀로 모델을 불러온 뒤 **11단계 실시간 판정**을 바로 실행할 수 있습니다.
(1~4단계로 보드에 연결은 되어 있어야 합니다.)

In [ ]:
# outputs 폴더의 저장된 모델 목록
out_dir = Path('..') / 'outputs'
saved = sorted(out_dir.glob('emg_model_*.joblib'))

if not saved:
    print('저장된 모델이 없습니다. 먼저 10단계로 저장하세요.')
else:
    print('저장된 모델:')
    for i, p in enumerate(saved):
        print(f'  [{i}] {p.name}')
    print()

    # 가장 최근 모델을 불러온다 (다른 것을 쓰려면 인덱스 변경)
    PICK = -1
    bundle = joblib.load(saved[PICK])

    clf      = bundle['model']
    GESTURES = bundle['gestures']
    FS       = bundle['fs']
    N_CH     = bundle['channels']
    WINDOW_SAMPLES = bundle['window']
    NOTCH_FREQ = bundle['filter']['notch']
    BAND       = bundle['filter']['band']

    # 필터 계수 재설계 (불러온 설정으로)
    _nyq = FS / 2
    b_notch, a_notch = sp_signal.iirnotch(NOTCH_FREQ / _nyq, Q=30)
    b_band,  a_band  = sp_signal.butter(4, [BAND[0]/_nyq, BAND[1]/_nyq], btype='band')

    print(f'불러온 모델: {saved[PICK].name}')
    print(f'  동작   : {list(GESTURES.values())}')
    print(f'  정확도 : {bundle["accuracy"]:.1f}%')
    print(f'  학습일 : {bundle["date"]}')
    print()
    print('이제 11단계 실시간 판정을 실행할 수 있습니다.')
    print('(4단계로 보드에 연결되어 있어야 합니다.)')

---
# 정리

## 오늘 배운 흐름

```
팔 근육 EMG (4채널, 200Hz)
      ↓  필터 (60Hz 노치 + 30-95Hz 대역통과)
      ↓  특징 추출 (채널별 포락선 평균 -> 숫자 4개)
캘리브레이션 (동작 × 40회)
      ↓
Random Forest 학습  →  모델 저장 (joblib)
      ↓
실시간 판정 (다수결 안정화)
```

## 핵심 개념

| 개념 | 의미 |
| --- | --- |
| EMG | 근육이 낼 때 나오는 전기 신호 (EEG 보다 높은 주파수) |
| 특징 추출 | 원신호를 판단에 쓸 핵심 숫자로 요약 |
| 포락선 | 신호의 세기 윤곽 (`|Hilbert|`) |
| Random Forest | 여러 결정 트리의 다수결 분류기 |
| 시간순 분할 | 정직한 성능 평가 (데이터 누수 방지) |
| 다수결 안정화 | 튀는 예측을 흡수해 안정된 판정 |

## 과제 아이디어

**기초**
1. 무작위 분할 vs 시간순 분할 정확도 비교 — 왜 차이가 나는가
2. `N_REPEAT` 를 20/40/80 으로 바꿔 정확도 변화 관찰
3. 트리 개수(`n_estimators`)를 바꿔 보기

**응용**
4. 지난주 저장 모델을 오늘 신호로 시험 — 정확도가 떨어지나?
5. 친구 모델로 내 동작 판정 — 개인차 확인
6. 동작을 4개로 늘리기 (예: 엄지척 추가)

**심화**
7. 특징을 늘려 보기 (포락선 평균 외에 표준편차, 최대값 등)
8. 다른 분류기(SVM, LogisticRegression)와 비교
9. 실시간 판정 결과로 아두이노 로봇 손 제어 (3차시 예고)

---

*재활재생개론 · OpenBCI Ganglion 실습 2*